In [1]:
import os

In [28]:
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling.datamodel.pipeline_options import PdfPipelineOptions
import tiktoken

In [29]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
embedding_model = "text-embedding-3-small"
dimension=1536

In [3]:
hr_ploicy_file=r"C:\dev\lv-assignment1\zdata\ABC_Corporation_HR_Policy_Manual.pdf"
law_policy_file=r"C:\dev\lv-assignment1\zdata\Historical_Court_Decisions_Compilation.pdf"

In [ ]:
def parse_document(file_path:str):

    pdf_option = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True)
    converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_option)})
    doc=converter.convert(hr_ploicy_file)
    return doc.document

hr_policy_doc=parse_document(hr_ploicy_file)
law_policy_doc=parse_document(law_policy_file)

In [13]:
def chunking(document):
    tiktoken_encoder = tiktoken.get_encoding("cl100k_base")
    tokenizer = OpenAITokenizer(tokenizer=tiktoken_encoder,max_tokens=512)
    chunker = HybridChunker(tokenizer=tokenizer, chunk_size=512, chunk_overlap=50,merge_peers=True)
    raw_chunks = list(chunker.chunk(document))
    return raw_chunks

In [15]:
def data_prep(chunks):

    """we are preparing data in the format required by VB"""
    result = []

    for idx,chunk in enumerate(chunks):
        headings=[]
        if chunk.meta.headings:
            headings=chunk.meta.headings
        page_no =[]
        if chunk.meta.doc_items:
            for item in chunk.meta.doc_items:
                for prov in item.prov:
                    page_no.append(prov.page_no)
        captions=[]
        if chunk.meta.captions:
            captions=chunk.meta.captions

        result.append(
            {
                "filename":chunk.meta.origin.filename,
                "headings":headings,
                "page_no":page_no,
                "captions":captions,
                "text":chunk.text
            }
        )
    return result



In [16]:
def embedding(chucks_cleaned):
    embedding_client = OpenAI(api_key=openai_api_key)
    embedding =[]
    for chunk in chucks_cleaned:
        response = embedding_client.embeddings.create(
        input=chunk["text"],
        model=embedding_model,
        encoding_format="float")
        embedding.append(response.data[0].embedding)
    return embedding

Pinecone Index creation and addition work

In [35]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_environment = os.getenv("PINECONE_ENVIRONMENT")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")


In [36]:
pinecone_index_name

'hr-policy'

In [37]:
from pinecone.grpc import PineconeGRPC

In [38]:
pc=PineconeGRPC(api_key=pinecone_api_key)

In [39]:
index_description = pc.describe_index(name=pinecone_index_name)

In [40]:
index_description.host

'hr-policy-rev8dya.svc.aped-4627-b74a.pinecone.io'

In [41]:
index = pc.Index(host=index_description.host)

In [133]:
for chunk, embed in zip(result, embedding):
    index.upsert(
        vectors=[
            {
                "id": f"{chunk['filename']}_{chunk['page_no']}_{chunk['headings']}",
                "values": embed,
                "metadata": {
                    "filename": chunk["filename"],
                    "headings": chunk["headings"],
                    "page_no": int(chunk["page_no"][0]) if isinstance(chunk["page_no"], list) else int(chunk["page_no"]),
                    "captions": chunk["captions"],
                    "text": chunk["text"]
                }
            }
        ]
    )

In [31]:
def HR_retrieval(question):
    embedding_client = OpenAI(api_key=openai_api_key)
    query_emded=embedding_client.embeddings.create(
        input=question,
        model=embedding_model,
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer = index.query(
        vector=query_vector,
        top_k=2,
        include_metadata=True
    )
    return answer.matches[0].metadata["text"]

In [42]:
HR_retrieval("when is the performance review done?")

'Performance reviews are conducted bi n annually. Employees receive structured feedback, goal alignment, and development planning. Outstanding performers may receive incentives and career advancement opportunities.'